# 🕵️ EDA – Fraudulent Job Postings Detection
**Dataset:** `fake_job_postings.csv` (Kaggle / EMSCAD)

**Goal:** Understand the data distribution, class imbalance, missing values, and text patterns
to inform feature engineering decisions before model training.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

plt.style.use('dark_background')
PALETTE = ['#43a047', '#e53935']   # green = genuine, red = fraudulent
print('Libraries loaded ✅')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('data/fake_job_postings.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Column names:')
print(df.columns.tolist())

In [ ]:
df.info()

## 3. Class Distribution (Target Variable)

In [ ]:
counts = df['fraudulent'].value_counts()
labels = ['Genuine (0)', 'Fraudulent (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=PALETTE, edgecolor='white', linewidth=0.5)
axes[0].set_title('Class Distribution – Count', fontsize=13, pad=12)
axes[0].set_ylabel('Number of Postings')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 80, str(v), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=PALETTE, startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':1})
axes[1].set_title('Class Distribution – Proportion', fontsize=13, pad=12)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n📌 Insight: Dataset is HIGHLY IMBALANCED.")
print(f"   Only ~{counts[1]/len(df)*100:.1f}% of postings are fraudulent.")
print("   → We must use stratified splitting and class-weighted / F1-focused training.")

## 4. Missing Values Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_df.index, y='Missing %', data=missing_df,
            palette='Reds_r', edgecolor='white')
plt.xticks(rotation=45, ha='right')
plt.title('Missing Value Percentage per Column', fontsize=13)
plt.ylabel('Missing (%)')
plt.tight_layout()
plt.show()

print("\n📌 Insight: Text columns (salary_range, department, company_profile, requirements,")
print("   benefits) have significant missing values.")
print("   → Fill with empty string before TF-IDF vectorization.")
print("   → Categorical NaN → impute with most-frequent value.")

## 5. Categorical Feature Analysis

In [ ]:
cat_cols = ['employment_type', 'required_experience', 'required_education']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    data = df.groupby([col, 'fraudulent']).size().unstack(fill_value=0)
    data.plot(kind='bar', ax=ax, color=PALETTE, edgecolor='white', linewidth=0.4)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(['Genuine', 'Fraudulent'], fontsize=8)

plt.tight_layout()
plt.show()

print("\n📌 Insight: 'Employment Type' missing in many fraudulent postings.")
print("   Fraudulent ads often skip required_experience / required_education.")

## 6. Binary Feature Analysis (Telecommuting, Logo, Questions)

In [ ]:
binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, binary_cols):
    fraud_rate = df.groupby(col)['fraudulent'].mean() * 100
    bars = ax.bar(fraud_rate.index.astype(str), fraud_rate.values,
                  color=['#e53935','#43a047'], edgecolor='white')
    ax.set_title(f'Fraud Rate by {col}', fontsize=10)
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_xlabel(col)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.2,
                f'{bar.get_height():.1f}%',
                ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📌 Insight:")
print("   - Postings WITHOUT a company logo → much higher fraud rate.")
print("   - Postings WITH screening questions → lower fraud rate.")
print("   → These binary features are strong signals for fraud detection.")

## 7. Text Length Analysis

In [ ]:
df['desc_length']    = df['description'].fillna('').apply(len)
df['req_length']     = df['requirements'].fillna('').apply(len)
df['profile_length'] = df['company_profile'].fillna('').apply(len)

length_cols = ['desc_length', 'req_length', 'profile_length']
titles      = ['Description Length', 'Requirements Length', 'Company Profile Length']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, title in zip(axes, length_cols, titles):
    for label, color, val in zip([0, 1], PALETTE, [0, 1]):
        subset = df[df['fraudulent'] == label][col]
        ax.hist(subset, bins=40, alpha=0.65, color=color,
                label='Genuine' if label == 0 else 'Fraudulent')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n📌 Insight:")
print("   - Fraudulent postings tend to have SHORTER descriptions and NO requirements.")
print("   - Genuine postings have richer text content.")
print("   → TF-IDF on combined text will capture these vocabulary differences.")

## 8. Word Clouds – Fraudulent vs Genuine

In [ ]:
genuine_text    = ' '.join(df[df['fraudulent']==0]['description'].fillna(''))
fraudulent_text = ' '.join(df[df['fraudulent']==1]['description'].fillna(''))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

wc_genuine = WordCloud(width=700, height=350, background_color='#0f1117',
                       colormap='Greens', max_words=100).generate(genuine_text)
axes[0].imshow(wc_genuine, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('✅ Genuine Job Postings – Top Words', fontsize=12, color='#43a047')

wc_fraud = WordCloud(width=700, height=350, background_color='#0f1117',
                     colormap='Reds', max_words=100).generate(fraudulent_text)
axes[1].imshow(wc_fraud, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('⚠️ Fraudulent Job Postings – Top Words', fontsize=12, color='#e53935')

plt.tight_layout()
plt.show()

print("\n📌 Insight:")
print("   - Genuine: 'experience', 'team', 'skills', 'company', 'position'")
print("   - Fraudulent: 'work', 'home', 'data', 'entry', 'online', 'earn'")
print("   → Fraudulent postings lean heavily on 'work from home', 'data entry',")
print("     'earn money online' phrases — classic scam signals.")

## 9. Top Industries / Functions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['industry', 'function']):
    top = df[df['fraudulent']==1][col].value_counts().head(10)
    ax.barh(top.index, top.values, color='#e53935', edgecolor='white')
    ax.set_title(f'Top 10 {col.title()} in Fraudulent Postings', fontsize=11)
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

print("\n📌 Insight: Oil & Energy, IT, and Marketing industries are most impersonated.")
print("   Administrative / customer-service functions dominate fraudulent listings.")

## 10. EDA Summary

| Finding | Impact on Modelling |
|---|---|
| Severe class imbalance (~4.8% fraud) | Use stratified split, `class_weight='balanced'`, F1 metric |
| High missing % in text fields | Fill NaN → empty string before TF-IDF |
| Fraudulent posts = shorter text | Text length is implicitly captured by TF-IDF term counts |
| `has_company_logo=0` → higher fraud | Include as binary feature |
| `has_questions=1` → lower fraud | Include as binary feature |
| Fraudulent vocab: 'data entry', 'work home' | TF-IDF bigrams will capture these |
| Missing categorical → impute | Use `SimpleImputer(most_frequent)` + OneHotEncoder |
